# 14. Retrieval holdout contract and seal validation

This notebook records the locked query-generalization contract for the existing 10-card, 327-chunk corpus. It deliberately contains no retrieval, embedding, morphology, ranking, or metric-execution cell. Do not run an evaluation until the data seal and a separately approved candidate-config seal both validate.


## Evaluation boundary

The 30 rows are 10 card-alias checks and 20 evidence checks. Card Hit@3 has denominator 30; evidence Strict Hit@3, Recall@5, MRR@5, and nDCG@5 have denominator 20. `holdout_contract.json` is authoritative for gates, invalid states, one-shot handling, and the Vector-versus-RRF selection rule.


In [ ]:
from __future__ import annotations

import hashlib
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
HOLDOUT = ROOT / 'notebooks' / 'data' / '14_retrieval_holdout'
CORPUS = ROOT / 'notebooks' / 'data' / '13_hierarchical_chunking_retrieval' / 'chunks.jsonl'

def normalized(value: str) -> str:
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', value)).strip()

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


In [ ]:
queries = json.loads((HOLDOUT / 'holdout_queries.json').read_text(encoding='utf-8'))['queries']
labels = json.loads((HOLDOUT / 'holdout_labels.json').read_text(encoding='utf-8'))['labels']
contract = json.loads((HOLDOUT / 'holdout_contract.json').read_text(encoding='utf-8'))
manifest = json.loads((HOLDOUT / 'holdout_data_manifest.json').read_text(encoding='utf-8'))
chunks = [json.loads(line) for line in CORPUS.read_text(encoding='utf-8').splitlines()]
chunk_by_id = {chunk['id']: chunk for chunk in chunks}
label_by_id = {label['query_id']: label for label in labels}

assert len(queries) == 30 == len(labels)
assert Counter(query['category'] for query in queries) == {'card_alias': 10, 'numeric_condition': 10, 'semantic_multi_condition': 10}
assert Counter(query['question_group'] for query in queries) == {'card': 10, 'evidence': 20}
assert Counter(label['expected_card'] for label in labels).values() and all(count == 3 for count in Counter(label['expected_card'] for label in labels).values())
assert len(chunks) == 327
for label in labels:
    for evidence in label['relevant_chunks']:
        chunk = chunk_by_id[evidence['chunk_id']]
        assert chunk['metadata']['card_key'] == label['expected_card']
        assert chunk['metadata']['level'] in label['allowed_levels']
        assert normalized(evidence['evidence_span']) in normalized(chunk['document'])
assert all(label_by_id[query['query_id']]['relevant_chunks'] for query in queries if query['question_group'] == 'evidence')


In [ ]:
for relative_path, expected_digest in manifest['file_sha256'].items():
    assert sha256(ROOT / relative_path) == expected_digest, relative_path
assert manifest['status'] == 'locked_not_evaluated'
assert manifest['candidate_config_seal']['status'] == 'pending'
print('Holdout data seal validates. Candidate configuration seal is still pending; do not evaluate.')


## Candidate seal checkpoint

After development-set selection, create a separate candidate seal before any holdout read for evaluation. It must identify the Vector baseline and exactly one RRF configuration and pin chunks, aliases, normalization, morphology, embedding model, BM25, RRF weights, tie-break, and rank depth. A failed hash or pre-execution infrastructure failure produces no result; a low score does not authorize a rerun or tuning against this folder.
